# SRΨ-Engine Ablation Study on Google Colab

## 🚀 Quick Start Guide

**Step 1**: Enable GPU (Runtime → Change runtime type → GPU → Save)

**Step 2**: Run each cell below in order (click ▶️ or press Shift+Enter)

**Step 3**: Choose your experiment in Step 5

---

## Available Experiments
- `srpsi_real`: SRΨ Real-only (Exp2)
- `srpsi_no_r`: SRΨ without Rhythm operator (Exp3)
- `conv_baseline`: Pure Convolution baseline (Exp4)
- `transformer_rel_pe`: Transformer with Relative Position Encoding (Exp5)

## Expected Time
- GPU: ~30-40 minutes per experiment
- CPU: ~13-15 hours per experiment

**Pro tip**: Open multiple Colab tabs to run experiments in parallel!

## Step 1: Check GPU Availability

In [ ]:
import torch
import os

print('=== GPU Status ===')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('\n✅ GPU is ready!')
else:
    print('\n⚠️ No GPU detected!')
    print('Please enable GPU: Runtime → Change runtime type → GPU')

## Step 2: Clone Repository

In [ ]:
# Clone the repository
!git clone https://github.com/Mozilla2004/srpsi-engine-tiny.git

# Change to project directory
%cd srpsi-engine-tiny

# List files
print('\n=== Repository cloned successfully ===')
!ls -la

## Step 3: Install Dependencies

In [ ]:
# Install required packages
!pip install -q pyyaml tqdm

import yaml
from tqdm import tqdm

print('\n=== Dependencies installed ===')
print('✓ pyyaml')
print('✓ tqdm')
print('✓ torch (already installed)')

## Step 4: Generate/Check Training Data

In [ ]:
# Check if data exists
import os

data_file = 'data/burgers_1d.npy'

if os.path.exists(data_file):
    size_mb = os.path.getsize(data_file) / (1024 * 1024)
    print(f'\n=== Data already exists ===')
    print(f'File: {data_file}')
    print(f'Size: {size_mb:.1f} MB')
    print('✓ Ready to train!')
else:
    print('\n=== Generating training data ===')
    print('This may take a few minutes...')
    !python src/data_gen.py
    
    if os.path.exists(data_file):
        size_mb = os.path.getsize(data_file) / (1024 * 1024)
        print(f'\n✓ Data generated: {size_mb:.1f} MB')
    else:
        print('✗ Data generation failed!')

## Step 5: Run Experiment

**Choose your experiment below by modifying `model_name`**

Available options:
- `srpsi_real`: SRΨ Real-only (Exp2)
- `srpsi_no_r`: SRΨ without Rhythm (Exp3)
- `conv_baseline`: Pure Convolution (Exp4)
- `transformer_rel_pe`: Transformer with Relative Position (Exp5)

In [ ]:
# ============================================================
# ⚙️ CONFIGURE YOUR EXPERIMENT HERE
# ============================================================

model_name = 'srpsi_real'  # Change this: srpsi_real, srpsi_no_r, conv_baseline, transformer_rel_pe
num_epochs = 80            # Number of training epochs (default: 80)

# ============================================================

print(f'\n=== Starting Experiment: {model_name} ===')
print(f'Model: {model_name}')
print(f'Epochs: {num_epochs}')
print(f'Output: outputs/ablation_{model_name}')
print('='*60)

# Run training
!python src/train.py \
  --config config/burgers.yaml \
  --model {model_name} \
  --data data/burgers_1d.npy \
  --output outputs/ablation_{model_name} \
  --epochs {num_epochs}

## Step 6: Check Training Results

In [ ]:
# Find and display best checkpoint
import os
from glob import glob

checkpoint_dir = f'outputs/ablation_{model_name}/{model_name}/checkpoints'

if os.path.exists(checkpoint_dir):
    checkpoints = glob(f'{checkpoint_dir}/best*.pt')
    
    if checkpoints:
        best_model = checkpoints[0]
        size_mb = os.path.getsize(best_model) / (1024 * 1024)
        print(f'\n=== Training Complete ===')
        print(f'✓ Best model: {best_model}')
        print(f'  Size: {size_mb:.2f} MB')
        
        # Also check for final model
        final_models = glob(f'{checkpoint_dir}/final*.pt')
        if final_models:
            print(f'✓ Final model: {final_models[0]}')
    else:
        print('\n⚠️ No best checkpoint found.')
        print('Training may still be in progress.')
        print('\nAll checkpoints:')
        !ls -lh {checkpoint_dir}/
else:
    print(f'\n✗ Checkpoint directory not found: {checkpoint_dir}')

## Step 7: Monitor Training Log

In [ ]:
# Display last 50 lines of training log
log_file = f'logs/ablation_{model_name}.log'

if os.path.exists(log_file):
    print(f'\n=== Training Log (last 50 lines) ===')
    !tail -50 {log_file}
else:
    print(f'\n⚠️ Log file not found: {log_file}')
    print('Training may not have started yet.')

## Step 8: Download Results

### Method 1: Download via Colab Interface
1. Click the 📁 folder icon on the left
2. Navigate to: `srpsi-engine-tiny/outputs/ablation_{model_name}/{model_name}/checkpoints/`
3. Right-click on `best*.pt` → Download

### Method 2: Download via Code

In [ ]:
from google.colab import files

# Download best checkpoint
checkpoint_path = f'outputs/ablation_{model_name}/{model_name}/checkpoints/best.pt'

if os.path.exists(checkpoint_path):
    print(f'Downloading: {checkpoint_path}')
    files.download(checkpoint_path)
else:
    print(f'File not found: {checkpoint_path}')
    print('\nAvailable files:')
    !ls -lh outputs/ablation_{model_name}/{model_name}/checkpoints/
    print('\nPlease download manually from the file browser on the left.')

## 📊 Quick Reference

### Run Multiple Experiments in Parallel

To run multiple experiments at the same time:

1. **Open this notebook in multiple browser tabs**

2. **In each tab, set a different `model_name`**:
   - Tab 1: `model_name = 'srpsi_real'`
   - Tab 2: `model_name = 'srpsi_no_r'`
   - Tab 3: `model_name = 'conv_baseline'`
   - Tab 4: `model_name = 'transformer_rel_pe'`

3. **Run Step 5 in each tab**

4. **All experiments will complete in ~30-40 minutes instead of 15 hours!**

### Training Time Estimates (GPU)

| Experiment | Time per Epoch | Total (80 epochs) |
|-----------|----------------|-------------------|
| SRΨ Real | ~20 sec | ~30 min |
| SRΨ w/o R | ~25 sec | ~35 min |
| Conv Baseline | ~15 sec | ~25 min |
| Transformer | ~30 sec | ~45 min |

### Troubleshooting

**Issue**: "No module named 'torch'"
- **Solution**: Make sure GPU is enabled (Runtime → Change runtime type → GPU)

**Issue**: "Out of memory"
- **Solution**: Restart runtime (Runtime → Restart runtime)

**Issue**: Training is slow
- **Solution**: Check GPU is actually being used (Step 1)